# N-CMAPSS Loaders — Phase 1 (Rewritten: Chunked, Never Silent)

Loads all 9 usable N-CMAPSS files from Google Drive and constructs the
5-flag multi-label fault labels, using the label-construction logic
already agreed on in `deliverables/relabel_decision_akindoju_alignment.docx`
and `deliverables/rul_and_fault_classification_approach.docx`:

- **Rule 1 ("when"):** a row is a failure row if `RUL <= 30` cycles.
- **Rule 2 ("what"):** for failure rows, turn on the fault flag(s) for
  whichever component(s) that row's source file is mapped to
  (Akindoju, 2025, Table 4.1).

## What changed from the first version

The first version of this notebook loaded all 9 files into memory at
once and concatenated them — roughly 20-25GB of RAM, which crashed a
standard Colab runtime **silently** (no error, just a runtime restart
that reset every cell). See `deliverables/pipeline_run_record.docx`
for the full story.

This version fixes both problems:
1. **Chunked processing** — each file is loaded, labeled, saved to
   disk, and freed from memory *one at a time*, via
   `process_and_save_all_files()`. Memory never holds more than one
   file's data (a few GB at most), never all 9 at once.
2. **Never silent** — every step prints as it happens, so a hang or
   crash is visible immediately instead of looking identical to a
   cell that's quietly working.

**Note:** the `T` group (engine health parameters) is never loaded
here — it's simulation-only ground truth that wouldn't exist on a real
aircraft, so it stays out of the model's input features (see
`docs/data-dictionary.md`, Sections 9-10). `Y` (RUL) *is* loaded, but
only to build the label — it is not saved as a model input either.

In [1]:
import os

import numpy as np
import h5py

## 1. Fixed constants

These constants **are** the label-construction logic. Nothing later in
this notebook should need to change to load a new N-CMAPSS file that's
already covered by Table 4.1.

In [2]:
# Rule 2 ("what"): file -> which component(s) that file was built to
# fail, straight from Akindoju (2025), Table 4.1. This dict is checked
# against the same 9-file, 60-unit total documented in
# docs/data-dictionary.md, so if that table ever changes, this must too.
FAULT_COMPONENT_MAP = {
    "N-CMAPSS_DS01-005.h5": ["hpt"],
    "N-CMAPSS_DS02-006.h5": ["hpt"],
    "N-CMAPSS_DS03-012.h5": ["hpt", "lpt"],
    "N-CMAPSS_DS04.h5": ["fan"],
    "N-CMAPSS_DS05.h5": ["hpc"],
    "N-CMAPSS_DS06.h5": ["hpc", "lpc"],
    "N-CMAPSS_DS07.h5": ["lpt"],
    "N-CMAPSS_DS08a-009.h5": ["fan", "lpc", "hpc", "hpt", "lpt"],
    "N-CMAPSS_DS08c-008.h5": ["fan", "lpc", "hpc", "hpt", "lpt"],
}

# The 9 usable files, in one fixed order -- every summary this notebook
# produces lists files in this same order.
NCMAPSS_FILES = list(FAULT_COMPONENT_MAP.keys())

# Fixed output-column order for the label matrix. Downstream code (the
# model's output layer, evaluation/metrics.py) can rely on column 0
# always being fan_fail, column 1 always lpc_fail, and so on.
FAULT_COLUMNS = ["fan_fail", "lpc_fail", "hpc_fail", "hpt_fail", "lpt_fail"]

# Rule 1 ("when"): the RUL cutoff, in cycles, below which a row counts
# as "failing." This number is a judgment call carried over from
# Akindoju (2025) / the original N-CMAPSS scoring paper, not a
# physical constant -- see rul_and_fault_classification_approach.docx.
RUL_FAILURE_THRESHOLD = 30

# Default location of the raw .h5 files once Google Drive is mounted,
# matching the path already used in notebooks/01_eda.ipynb.
DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/raw"

# Where processed (per-file, labeled) output gets saved -- matches the
# repo's existing data/processed/ folder convention.
DEFAULT_PROCESSED_DIR = "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/processed"

## 2. Mount Google Drive

Run this cell once per Colab session, before any cell that reads from
or writes to Drive. It's safe to re-run — Colab just re-confirms an
already-mounted Drive. Notice it prints what it's doing, rather than
running silently.

In [3]:
def mount_google_drive():
    """Mount Google Drive inside a Colab runtime."""
    try:
        # This import only succeeds inside a real Colab runtime, so it
        # doubles as the "are we in Colab?" check.
        from google.colab import drive
    except ImportError:
        # Not in Colab (e.g. running this notebook locally) -- there is
        # no Drive to mount, so do nothing rather than raise an error.
        print("Not running in Colab -- skipping Drive mount.")
        return

    print("Mounting Google Drive ...")
    drive.mount("/content/drive")  # triggers Colab's one-time browser auth prompt
    print("Drive mounted.")


mount_google_drive()

Mounting Google Drive ...
Mounted at /content/drive
Drive mounted.


## 3. Build the label matrix for one file

This is Rules 1 and 2 together, applied to a single file's RUL column.
Unchanged from the first version — this logic was already verified and
is not what caused the crash.

In [4]:
def _build_labels(rul, fault_components):
    """Construct the 5-column binary fault-flag matrix for one file.

    Parameters
    ----------
    rul : np.ndarray, shape (n_rows,)
        The RUL value for every row in one file (from that file's Y
        group), used only here to decide failure vs. healthy.
    fault_components : list[str]
        This file's mapped component(s), e.g. ["hpt", "lpt"] for DS03
        -- looked up from FAULT_COMPONENT_MAP by the caller.

    Returns
    -------
    np.ndarray, shape (n_rows, 5), dtype int8
        1 = that component is flagged failing on that row, 0 = not.
        Column order matches FAULT_COLUMNS.
    """
    n_rows = rul.shape[0]

    # Start every row as all-zero (healthy on all 5 flags) -- Rule 2
    # only ever turns flags ON below, it never needs to turn any off.
    labels = np.zeros((n_rows, len(FAULT_COLUMNS)), dtype=np.int8)

    # Rule 1 ("when"): True for every row within RUL_FAILURE_THRESHOLD
    # cycles of end-of-life; False (healthy) for everything else.
    is_failure_row = rul <= RUL_FAILURE_THRESHOLD

    # Rule 2 ("what"): for THIS file's mapped component(s) only, set
    # that column to 1 on every failure row. A file mapped to more than
    # one component (e.g. DS03 -> hpt + lpt) sets more than one column,
    # which is exactly how one row ends up with two flags on at once.
    for component in fault_components:
        col_index = FAULT_COLUMNS.index(f"{component}_fail")  # e.g. "hpt" -> "hpt_fail" -> column 3
        labels[is_failure_row, col_index] = 1  # only failure rows get touched; healthy rows stay 0

    return labels

## 4. Load one file

Reads `W`, `X_s`, `X_v`, `A`, `Y` from one `.h5` file and attaches the
label matrix built above. This is the ONLY function that reads raw
HDF5, and it handles exactly one file at a time by design.

In [5]:
def load_file(filename, data_dir=DEFAULT_DRIVE_DIR, split="dev"):
    """Load one N-CMAPSS file's model inputs and construct its labels.

    Parameters
    ----------
    filename : str
        Must be one of the 9 keys in FAULT_COMPONENT_MAP.
    data_dir : str
        Directory the raw .h5 files live in. Defaults to the Google
        Drive path; pass a local folder instead for testing.
    split : str
        "dev" or "test" -- N-CMAPSS stores each group twice, suffixed
        _dev / _test (e.g. "W_dev", "W_test").

    Returns
    -------
    dict with keys "W", "X_s", "X_v", "A", "Y", "labels" -- numpy
    arrays, all with the same row count, row-aligned to each other.
    """
    if filename not in FAULT_COMPONENT_MAP:
        # Fail loudly instead of silently loading a file with no known
        # fault-component mapping -- there would be no valid label for it.
        raise ValueError(f"{filename!r} is not one of the 9 mapped files in Table 4.1.")

    path = f"{data_dir}/{filename}"  # build the full path once, reused for every group below

    with h5py.File(path, "r") as f:
        # [:] copies each group out of the HDF5 file into a normal numpy
        # array -- without it, `f[...]` stays a lazy on-disk reference.
        w = f[f"W_{split}"][:]      # flight-condition descriptors (4 cols) -- model input
        x_s = f[f"X_s_{split}"][:]  # physical sensor measurements (14 cols) -- model input
        x_v = f[f"X_v_{split}"][:]  # virtual/derived sensors (14 cols) -- model input (practical compromise)
        a = f[f"A_{split}"][:]      # unit, cycle, Fc, hs -- identifying columns, not a model input
        y = f[f"Y_{split}"][:]      # RUL -- used ONLY below to build the label, never a model input

    rul = y.reshape(-1)  # Y comes back as an (n_rows, 1) column; flatten to 1-D for the threshold check
    labels = _build_labels(rul, FAULT_COMPONENT_MAP[filename])  # apply Rules 1 + 2 for this file

    return {"W": w, "X_s": x_s, "X_v": x_v, "A": a, "Y": y, "labels": labels}

## 5. Process and save all 9 files — ONE AT A TIME

**This replaces the old `load_all_files()`.** Instead of holding all 9
files in memory and concatenating them, this loads one file, saves it
to disk immediately, then frees that file's memory before moving to
the next. Peak memory is capped at roughly one file's size (a few GB),
never all 9 at once. Every step prints, so progress — or a stall — is
always visible.

In [6]:
def process_and_save_all_files(data_dir=DEFAULT_DRIVE_DIR, out_dir=DEFAULT_PROCESSED_DIR, split="dev", files=None):
    """Load, label, and save all 9 files to disk ONE AT A TIME.

    Parameters
    ----------
    data_dir : str
        Directory the raw .h5 files live in.
    out_dir : str
        Directory to save the processed, labeled .npz files into.
        Created automatically if it doesn't already exist.
    split : str
        "dev" or "test".
    files : list[str] or None
        Which files to process, in order. Defaults to all 9
        (NCMAPSS_FILES). Overridable for a quick test on 1-2 files.

    Returns
    -------
    list[dict]
        One small summary dict per file -- {"filename", "n_rows",
        "output_path", "output_mb"} -- NOT the actual data arrays, so
        this function's own return value never grows large no matter
        how many files are processed.
    """
    files = files if files is not None else NCMAPSS_FILES  # default to the full documented 9-file scope

    os.makedirs(out_dir, exist_ok=True)  # create data/processed/ (or the given out_dir) if it isn't there yet

    summaries = []          # one small dict per file -- never holds the actual arrays
    running_total_rows = 0  # accumulated as a plain int, not by keeping every file's arrays around

    for i, filename in enumerate(files, start=1):
        print(f"[{i}/{len(files)}] Loading {filename} ...")  # printed BEFORE the slow step, so a hang is visible here

        loaded = load_file(filename, data_dir=data_dir, split=split)  # only THIS file's arrays are in memory now
        n_rows = loaded["W"].shape[0]
        print(f"    read {n_rows:,} rows")

        out_name = filename.replace(".h5", f"_{split}.npz")  # e.g. "N-CMAPSS_DS05.h5" -> "N-CMAPSS_DS05_dev.npz"
        out_path = os.path.join(out_dir, out_name)

        # savez_compressed writes all 6 arrays into one .npz file on disk.
        # This is the step that lets us free loaded's memory right after --
        # nothing later needs to keep this file's arrays in RAM.
        np.savez_compressed(out_path, **loaded)
        output_mb = os.path.getsize(out_path) / (1024 * 1024)
        print(f"    saved to {out_path} ({output_mb:.1f} MB)")

        summaries.append({
            "filename": filename,
            "n_rows": n_rows,
            "output_path": out_path,
            "output_mb": output_mb,
        })
        running_total_rows += n_rows

        del loaded  # explicitly drop the reference so this file's ~1-4GB is freed before the next loop iteration

    print(f"Done -- {len(files)} files processed, {running_total_rows:,} total rows saved to {out_dir}")
    return summaries

## 6. Read processed files back — one at a time or streamed

Two small helpers for using the saved `.npz` files afterward, without
ever needing to re-read the original HDF5 or hold all 9 files in
memory again.

In [7]:
def load_processed_file(filename, out_dir=DEFAULT_PROCESSED_DIR, split="dev"):
    """Load ONE already-processed file back from its saved .npz."""
    out_name = filename.replace(".h5", f"_{split}.npz")
    out_path = os.path.join(out_dir, out_name)
    with np.load(out_path) as npz:
        # dict(npz) copies each array out of the open .npz handle --
        # after this, the file itself can be closed (the `with` block
        # above does that automatically) without losing the data.
        return {key: npz[key] for key in npz.files}


def iter_processed_files(out_dir=DEFAULT_PROCESSED_DIR, split="dev", files=None):
    """Yield one processed file's data at a time (the "chunked" read pattern).

    Example: for filename, file_data in iter_processed_files(): ...
    file_data goes out of scope and is freed before the next
    iteration's data is loaded -- this never holds all 9 files at once.
    """
    files = files if files is not None else NCMAPSS_FILES
    for filename in files:
        yield filename, load_processed_file(filename, out_dir=out_dir, split=split)

## 7. Run it

Processes and saves all 9 files from Drive, one at a time, printing
progress as it goes. This is the step that crashed silently in the
first version — it should now either visibly progress through all 9
files or fail with a real, visible error, never just "do nothing."

**This cell needs Drive mounted and the 9 files actually present** at
`DEFAULT_DRIVE_DIR` — skip it (and use the self-check in Section 8
instead) if you're just reviewing the logic without Drive access.

In [8]:
summaries = process_and_save_all_files()
summaries

[1/9] Loading N-CMAPSS_DS01-005.h5 ...
    read 4,906,636 rows
    saved to /content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/processed/N-CMAPSS_DS01-005_dev.npz (982.9 MB)
[2/9] Loading N-CMAPSS_DS02-006.h5 ...
    read 5,263,447 rows
    saved to /content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/processed/N-CMAPSS_DS02-006_dev.npz (1049.4 MB)
[3/9] Loading N-CMAPSS_DS03-012.h5 ...
    read 5,571,277 rows
    saved to /content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/processed/N-CMAPSS_DS03-012_dev.npz (1116.7 MB)
[4/9] Loading N-CMAPSS_DS04.h5 ...
    read 6,377,452 rows
    saved to /content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/processed/N-CMAPSS_DS04_dev.npz (1296.1 MB)
[5/9] Loading N-CMAPSS_DS05.h5 ...
    read 4,350,606 rows
    saved to /content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/processed/N-CMAPSS_DS05_dev.npz (870.9 MB)
[6/9] Loading N-CMAPSS_DS06.h5 ...
    read 4,257,209 rows
    saved to /content/drive/M

[{'filename': 'N-CMAPSS_DS01-005.h5',
  'n_rows': 4906636,
  'output_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/processed/N-CMAPSS_DS01-005_dev.npz',
  'output_mb': 982.9324493408203},
 {'filename': 'N-CMAPSS_DS02-006.h5',
  'n_rows': 5263447,
  'output_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/processed/N-CMAPSS_DS02-006_dev.npz',
  'output_mb': 1049.415090560913},
 {'filename': 'N-CMAPSS_DS03-012.h5',
  'n_rows': 5571277,
  'output_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/processed/N-CMAPSS_DS03-012_dev.npz',
  'output_mb': 1116.7338933944702},
 {'filename': 'N-CMAPSS_DS04.h5',
  'n_rows': 6377452,
  'output_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/processed/N-CMAPSS_DS04_dev.npz',
  'output_mb': 1296.0731840133667},
 {'filename': 'N-CMAPSS_DS05.h5',
  'n_rows': 4350606,
  'output_path': '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/processed/N-CMAPSS_DS05

## 8. Optional self-check (no Drive needed)

Builds two tiny synthetic `.h5` files, runs them through
`process_and_save_all_files()`, reloads one with `load_processed_file()`,
and streams both back with `iter_processed_files()` — confirming the
full chunked save/reload pipeline, plus the same label logic checked in
`label_construction_explained.docx`.

In [9]:
import tempfile

def _self_check():
    # --- Unit-level checks on _build_labels (unchanged logic) ---
    labels = _build_labels(np.array([145]), FAULT_COMPONENT_MAP["N-CMAPSS_DS05.h5"])
    assert labels.tolist() == [[0, 0, 0, 0, 0]], labels

    labels = _build_labels(np.array([12]), FAULT_COMPONENT_MAP["N-CMAPSS_DS03-012.h5"])
    assert labels.tolist() == [[0, 0, 0, 1, 1]], labels

    labels = _build_labels(np.array([30, 31]), FAULT_COMPONENT_MAP["N-CMAPSS_DS04.h5"])
    assert labels.tolist() == [[1, 0, 0, 0, 0], [0, 0, 0, 0, 0]], labels

    # --- End-to-end check on the new chunked save/reload pipeline ---
    with tempfile.TemporaryDirectory() as data_dir, tempfile.TemporaryDirectory() as out_dir:
        specs = {
            "N-CMAPSS_DS05.h5": np.array([[145], [20]]),        # DS05 -> hpc; 1 healthy, 1 hpc-fail
            "N-CMAPSS_DS04.h5": np.array([[5], [100], [30]]),   # DS04 -> fan; 2 fail, 1 healthy
        }
        for fname, y in specs.items():
            n = y.shape[0]
            with h5py.File(os.path.join(data_dir, fname), "w") as f:
                f.create_dataset("W_dev", data=np.random.rand(n, 4))
                f.create_dataset("X_s_dev", data=np.random.rand(n, 14))
                f.create_dataset("X_v_dev", data=np.random.rand(n, 14))
                f.create_dataset("A_dev", data=np.random.rand(n, 4))
                f.create_dataset("Y_dev", data=y)

        test_files = list(specs.keys())
        summaries = process_and_save_all_files(data_dir=data_dir, out_dir=out_dir, split="dev", files=test_files)
        assert summaries[0]["n_rows"] == 2 and summaries[1]["n_rows"] == 3
        assert all(os.path.exists(s["output_path"]) for s in summaries)

        reloaded = load_processed_file("N-CMAPSS_DS05.h5", out_dir=out_dir, split="dev")
        assert reloaded["labels"].tolist() == [[0, 0, 0, 0, 0], [0, 0, 1, 0, 0]], reloaded["labels"]

        total = sum(data["W"].shape[0] for _, data in iter_processed_files(out_dir=out_dir, split="dev", files=test_files))
        assert total == 5

    print("ALL SELF-CHECKS PASSED")


_self_check()

[1/2] Loading N-CMAPSS_DS05.h5 ...
    read 2 rows
    saved to /tmp/tmpgf2njlz2/N-CMAPSS_DS05_dev.npz (0.0 MB)
[2/2] Loading N-CMAPSS_DS04.h5 ...
    read 3 rows
    saved to /tmp/tmpgf2njlz2/N-CMAPSS_DS04_dev.npz (0.0 MB)
Done -- 2 files processed, 5 total rows saved to /tmp/tmpgf2njlz2
ALL SELF-CHECKS PASSED
